# 05. baseline 이상탐지 모델

이 노트북은 04번에서 고정한 feature 계약을 받아 `IsolationForest` 기반 baseline 이상탐지 모델을 학습한다.

사용 모델:
- `IsolationForest` anomaly baseline

평가 기준:
- `split_time_based`
- `split_substation_based`

추가 산출물:
- 모델/스케일러 저장 파일
- threshold sweep metrics
- 06번 LightGBM 위험도 모델에서 사용할 `anomaly_score`


In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, precision_recall_fscore_support, roc_auc_score
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('pyproject.toml을 찾을 수 없습니다. 프로젝트 루트에서 실행해 주세요.')


PROJECT_ROOT = find_project_root(Path.cwd())
FEATURE_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_features'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_baseline'
MODEL_DIR = OUTPUT_DIR / 'models'

INPUT_PATH = FEATURE_DIR / 'trainable_windows.csv'
FEATURE_COLUMNS_PATH = FEATURE_DIR / 'feature_columns.csv'
METADATA_COLUMNS_PATH = FEATURE_DIR / 'metadata_columns.csv'
IMPUTATION_VALUES_PATH = FEATURE_DIR / 'imputation_values.csv'

SCORES_PATH = OUTPUT_DIR / 'anomaly_baseline_scores.csv'
METRICS_PATH = OUTPUT_DIR / 'anomaly_baseline_metrics.csv'
THRESHOLDS_PATH = OUTPUT_DIR / 'anomaly_baseline_thresholds.csv'
THRESHOLD_SWEEP_METRICS_PATH = OUTPUT_DIR / 'anomaly_baseline_threshold_sweep_metrics.csv'
MODEL_METADATA_PATH = MODEL_DIR / 'baseline_model_metadata.json'
SCALER_PATH = MODEL_DIR / 'standard_scaler.joblib'
IFOREST_MODEL_PATH = MODEL_DIR / 'isolation_forest.joblib'

PRIMARY_SPLIT_COLUMN = 'split_time_based'
EVALUATION_SPLIT_COLUMNS = ['split_time_based', 'split_substation_based']
SPLIT_VALUES = ['train', 'validation', 'holdout']
THRESHOLD_QUANTILES = [0.95, 0.975, 0.99]
DEFAULT_THRESHOLD_QUANTILE = 0.99
RANDOM_STATE = 42
MODEL_VERSION = 'baseline_05_v1'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'입력 파일: {INPUT_PATH}')
print(f'출력 폴더: {OUTPUT_DIR}')
print(f'모델 저장 폴더: {MODEL_DIR}')
print(f'평가 split 컬럼: {EVALUATION_SPLIT_COLUMNS}')


입력 파일: C:\Project3\HeatGrid_Agent\data\processed\ml_features\trainable_windows.csv
출력 폴더: C:\Project3\HeatGrid_Agent\data\processed\ml_baseline
모델 저장 폴더: C:\Project3\HeatGrid_Agent\data\processed\ml_baseline\models
평가 split 컬럼: ['split_time_based', 'split_substation_based']


In [2]:
trainable_windows = pd.read_csv(INPUT_PATH)
feature_columns_table = pd.read_csv(FEATURE_COLUMNS_PATH)
metadata_columns_table = pd.read_csv(METADATA_COLUMNS_PATH)
imputation_values_table = pd.read_csv(IMPUTATION_VALUES_PATH)

metadata_columns = metadata_columns_table['column_name'].tolist()
selected_feature_columns = feature_columns_table.loc[
    feature_columns_table['selected_for_baseline'], 'column_name'
].tolist()

print('trainable_windows 크기:', trainable_windows.shape)
print('선택된 feature 수:', len(selected_feature_columns))
print('metadata 컬럼 수:', len(metadata_columns))
display(trainable_windows[['label', PRIMARY_SPLIT_COLUMN, 'split_substation_based']].head())
display(trainable_windows['label'].value_counts(dropna=False).rename_axis('라벨').reset_index(name='행 수'))


trainable_windows 크기: (2526, 232)
선택된 feature 수: 195
metadata 컬럼 수: 37


,label,split_time_based,split_substation_based
0,normal,validation,holdout
1,normal,validation,holdout
2,normal,validation,holdout
3,normal,validation,holdout
4,normal,validation,holdout


,라벨,행 수
0,normal,1767
1,pre_fault,759


## 1. train / validation / holdout 분리

두 anomaly baseline의 학습에는 다음 subset만 사용한다.
- `split_time_based == train`
- `label == normal`

평가는 `split_time_based`, `split_substation_based` 두 기준을 모두 저장한다.


In [3]:
train_df = trainable_windows.loc[trainable_windows[PRIMARY_SPLIT_COLUMN] == 'train'].copy()
validation_df = trainable_windows.loc[trainable_windows[PRIMARY_SPLIT_COLUMN] == 'validation'].copy()
holdout_df = trainable_windows.loc[trainable_windows[PRIMARY_SPLIT_COLUMN] == 'holdout'].copy()

train_normal_df = train_df.loc[train_df['label'] == 'normal'].copy()

if train_normal_df.empty:
    raise ValueError('train split에 normal 행이 없습니다.')

split_summary_rows = []
for split_column in EVALUATION_SPLIT_COLUMNS:
    for split_value in SPLIT_VALUES:
        split_df = trainable_windows.loc[trainable_windows[split_column] == split_value]
        split_summary_rows.append({
            'split_column': split_column,
            'split': split_value,
            '행 수': len(split_df),
            'normal': int((split_df['label'] == 'normal').sum()),
            'pre_fault': int((split_df['label'] == 'pre_fault').sum()),
        })
split_summary_rows.append({
    'split_column': 'training_subset',
    'split': 'time_train_normal_only',
    '행 수': len(train_normal_df),
    'normal': len(train_normal_df),
    'pre_fault': 0,
})

split_summary = pd.DataFrame(split_summary_rows)
display(split_summary)


,split_column,split,행 수,normal,pre_fault
0,split_time_based,train,1770,1263,507
1,split_time_based,validation,362,243,119
2,split_time_based,holdout,394,261,133
3,split_substation_based,train,2021,1380,641
4,split_substation_based,validation,249,196,53
5,split_substation_based,holdout,256,191,65
6,training_subset,time_train_normal_only,1263,1263,0


In [4]:
X_train_normal = train_normal_df[selected_feature_columns].to_numpy(dtype=float)
X_all = trainable_windows[selected_feature_columns].to_numpy(dtype=float)

scaler = StandardScaler()
X_train_normal_scaled = scaler.fit_transform(X_train_normal)
X_all_scaled = scaler.transform(X_all)

print('정상 train 입력 크기:', X_train_normal_scaled.shape)
print('전체 입력 크기:', X_all_scaled.shape)


정상 train 입력 크기: (1263, 195)
전체 입력 크기: (2526, 195)


## 2. baseline 모델 학습

- `IsolationForest`를 normal window 기준으로 학습한다.
- 모델 학습은 `split_time_based == train`의 normal 행만 사용한다.
- 산출된 `iforest_anomaly_score`는 06번 LightGBM 위험도 모델의 입력 feature로 사용한다.


In [5]:
isolation_forest = IsolationForest(
    n_estimators=300,
    contamination='auto',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
isolation_forest.fit(X_train_normal_scaled)

isolation_raw_score = -isolation_forest.score_samples(X_all_scaled)

train_normal_mask = trainable_windows[PRIMARY_SPLIT_COLUMN].eq('train') & trainable_windows['label'].eq('normal')
threshold_lookup = {}
train_normal_scores = isolation_raw_score[train_normal_mask]
for quantile in THRESHOLD_QUANTILES:
    threshold_lookup[('isolation_forest', quantile)] = float(np.quantile(train_normal_scores, quantile))

isolation_threshold = threshold_lookup[('isolation_forest', DEFAULT_THRESHOLD_QUANTILE)]

print('IsolationForest 기본 threshold:', isolation_threshold)


IsolationForest 기본 threshold: 0.5357784270858823


## 3. 윈도우별 score 테이블 생성


In [6]:
scores_df = trainable_windows[metadata_columns].copy()
scores_df['target_binary'] = (trainable_windows['label'] == 'pre_fault').astype(int)

scores_df['iforest_anomaly_score'] = isolation_raw_score
scores_df['anomaly_score'] = scores_df['iforest_anomaly_score']
scores_df['iforest_threshold'] = isolation_threshold
scores_df['anomaly_threshold'] = scores_df['iforest_threshold']
scores_df['iforest_anomaly_label'] = (scores_df['iforest_anomaly_score'] >= isolation_threshold).astype(int)
scores_df['anomaly_label'] = scores_df['iforest_anomaly_label']

display(scores_df.head())


,manufacturer,substation_id,source_file,window_start,window_end,main_missing_sensors,main_changed_sensors,season_bucket,label,fault_label,fault_event_id,estimated_lead_time_hours,normal_event_related,maintenance_related,disturbance_count,leakage_blocked_fault_count,window_source_type,use_for_supervised_training,configuration_type,normal_reference_group,s_hc1_control_unit_mode__dominant,s_dhw_control_unit_mode__dominant,s_hc1_heating_pump_status_setpoint__dominant,s_dhw_3-way_valve_status__dominant,s_hc1.2_heating_pump_status__dominant,s_hc1.3_heating_pump_status__dominant,s_hc1.1_control_unit_mode__dominant,s_hc1.2_control_unit_mode__dominant,s_hc1.2_dhw_control unit_mode__dominant,s_hc1.1_heating_pump_status__dominant,s_hc1.3_control_unit_mode__dominant,split_time_based,split_substation_based,split_regime_based,normal_reference_outlier,normal_reference_outlier_count,normal_reference_filter_reason,target_binary,iforest_anomaly_score,anomaly_score,iforest_threshold,anomaly_threshold,iforest_anomaly_label,anomaly_label
0,manufacturer 1,1,substation_1.csv,2019-12-01 00:00:00,2019-12-01 06:00:00,NaN,p_net_meter_energy|p_net_meter_flow|p_net_supp...,winter,normal,NaN,NaN,NaN,True,False,0,0,normal_context,True,SH + DHW,manufacturer 1|SH + DHW|winter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,validation,holdout,train,False,0,NaN,0,0.416895,0.416895,0.535778,0.535778,0,0
1,manufacturer 1,1,substation_1.csv,2019-12-01 06:00:00,2019-12-01 12:00:00,NaN,p_net_meter_flow|p_net_meter_energy|p_net_mete...,winter,normal,NaN,NaN,NaN,True,False,0,0,normal_context,True,SH + DHW,manufacturer 1|SH + DHW|winter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,validation,holdout,train,False,0,NaN,0,0.430012,0.430012,0.535778,0.535778,0,0
2,manufacturer 1,1,substation_1.csv,2019-12-01 18:00:00,2019-12-02 00:00:00,NaN,p_net_meter_flow|p_net_meter_energy|p_net_mete...,winter,normal,NaN,NaN,NaN,True,False,0,0,normal_context,True,SH + DHW,manufacturer 1|SH + DHW|winter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,validation,holdout,train,False,0,NaN,0,0.426032,0.426032,0.535778,0.535778,0,0
3,manufacturer 1,1,substation_1.csv,2019-12-02 12:00:00,2019-12-02 18:00:00,NaN,p_net_meter_flow|p_net_meter_energy|p_net_mete...,winter,normal,NaN,NaN,NaN,True,False,0,0,normal_context,True,SH + DHW,manufacturer 1|SH + DHW|winter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,validation,holdout,validation,False,0,NaN,0,0.440612,0.440612,0.535778,0.535778,0,0
4,manufacturer 1,1,substation_1.csv,2019-12-02 18:00:00,2019-12-03 00:00:00,NaN,p_net_meter_flow|p_net_meter_energy|p_net_mete...,winter,normal,NaN,NaN,NaN,True,False,0,0,normal_context,True,SH + DHW,manufacturer 1|SH + DHW|winter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,validation,holdout,validation,False,0,NaN,0,0.424733,0.424733,0.535778,0.535778,0,0


## 4. split별 기본 평가와 threshold sweep

평가 항목:
- ROC-AUC
- Average Precision
- threshold 기준 Precision / Recall / F1
- normal 구간 false positive rate

`split_time_based`와 `split_substation_based` 기준을 모두 계산한다.


In [7]:
def safe_roc_auc(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_score))


def safe_average_precision(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(average_precision_score(y_true, y_score))


def false_positive_rate(y_true, y_pred):
    negative_mask = y_true == 0
    if negative_mask.sum() == 0:
        return np.nan
    return float(((y_pred == 1) & negative_mask).sum() / negative_mask.sum())


MODEL_SCORE_SPECS = [
    ('isolation_forest', 'iforest_anomaly_score', 'iforest_anomaly_label'),
]

metric_rows = []
threshold_sweep_rows = []
for split_column in EVALUATION_SPLIT_COLUMNS:
    for split_name in SPLIT_VALUES:
        split_df = scores_df.loc[scores_df[split_column] == split_name].copy()
        y_true = split_df['target_binary'].to_numpy()

        for model_name, score_column, pred_column in MODEL_SCORE_SPECS:
            y_score = split_df[score_column].to_numpy()

            for quantile in THRESHOLD_QUANTILES:
                threshold_value = threshold_lookup[(model_name, quantile)]
                y_pred_at_quantile = (y_score >= threshold_value).astype(int)
                precision, recall, f1, _ = precision_recall_fscore_support(
                    y_true,
                    y_pred_at_quantile,
                    average='binary',
                    zero_division=0,
                )
                row = {
                    'evaluation_split_column': split_column,
                    'split': split_name,
                    'model_name': model_name,
                    'threshold_quantile': quantile,
                    'threshold_value': threshold_value,
                    'row_count': len(split_df),
                    'normal_count': int((split_df['label'] == 'normal').sum()),
                    'pre_fault_count': int((split_df['label'] == 'pre_fault').sum()),
                    'roc_auc': safe_roc_auc(y_true, y_score),
                    'average_precision': safe_average_precision(y_true, y_score),
                    'precision': float(precision),
                    'recall': float(recall),
                    'f1': float(f1),
                    'false_positive_rate': false_positive_rate(y_true, y_pred_at_quantile),
                    'score_column': score_column,
                }
                threshold_sweep_rows.append(row)
                if quantile == DEFAULT_THRESHOLD_QUANTILE:
                    metric_rows.append({**row, 'prediction_column': pred_column})

metrics_df = pd.DataFrame(metric_rows)
threshold_sweep_metrics_df = pd.DataFrame(threshold_sweep_rows)

display(metrics_df)
display(threshold_sweep_metrics_df.head(20))


,evaluation_split_column,split,model_name,threshold_quantile,threshold_value,row_count,normal_count,pre_fault_count,roc_auc,average_precision,precision,recall,f1,false_positive_rate,score_column,prediction_column
0,split_time_based,train,isolation_forest,0.99,0.535778,1770,1263,507,0.615898,0.407077,0.617647,0.041420,0.077634,0.010293,iforest_anomaly_score,iforest_anomaly_label
1,split_time_based,validation,isolation_forest,0.99,0.535778,362,243,119,0.612754,0.467153,0.000000,0.000000,0.000000,0.000000,iforest_anomaly_score,iforest_anomaly_label
2,split_time_based,holdout,isolation_forest,0.99,0.535778,394,261,133,0.715236,0.692980,1.000000,0.127820,0.226667,0.000000,iforest_anomaly_score,iforest_anomaly_label
3,split_substation_based,train,isolation_forest,0.99,0.535778,2021,1380,641,0.652096,0.504002,0.900000,0.056162,0.105727,0.002899,iforest_anomaly_score,iforest_anomaly_label
4,split_substation_based,validation,isolation_forest,0.99,0.535778,249,196,53,0.521948,0.230833,0.000000,0.000000,0.000000,0.025510,iforest_anomaly_score,iforest_anomaly_label
5,split_substation_based,holdout,isolation_forest,0.99,0.535778,256,191,65,0.555618,0.330451,0.333333,0.030769,0.056338,0.020942,iforest_anomaly_score,iforest_anomaly_label


,evaluation_split_column,split,model_name,threshold_quantile,threshold_value,row_count,normal_count,pre_fault_count,roc_auc,average_precision,precision,recall,f1,false_positive_rate,score_column
0,split_time_based,train,isolation_forest,0.950,0.516082,1770,1263,507,0.615898,0.407077,0.492063,0.122288,0.195893,0.050673,iforest_anomaly_score
1,split_time_based,train,isolation_forest,0.975,0.524479,1770,1263,507,0.615898,0.407077,0.549296,0.076923,0.134948,0.025337,iforest_anomaly_score
2,split_time_based,train,isolation_forest,0.990,0.535778,1770,1263,507,0.615898,0.407077,0.617647,0.041420,0.077634,0.010293,iforest_anomaly_score
3,split_time_based,validation,isolation_forest,0.950,0.516082,362,243,119,0.612754,0.467153,0.000000,0.000000,0.000000,0.000000,iforest_anomaly_score
4,split_time_based,validation,isolation_forest,0.975,0.524479,362,243,119,0.612754,0.467153,0.000000,0.000000,0.000000,0.000000,iforest_anomaly_score
5,split_time_based,validation,isolation_forest,0.990,0.535778,362,243,119,0.612754,0.467153,0.000000,0.000000,0.000000,0.000000,iforest_anomaly_score
6,split_time_based,holdout,isolation_forest,0.950,0.516082,394,261,133,0.715236,0.692980,1.000000,0.150376,0.261438,0.000000,iforest_anomaly_score
7,split_time_based,holdout,isolation_forest,0.975,0.524479,394,261,133,0.715236,0.692980,1.000000,0.135338,0.238411,0.000000,iforest_anomaly_score
8,split_time_based,holdout,isolation_forest,0.990,0.535778,394,261,133,0.715236,0.692980,1.000000,0.127820,0.226667,0.000000,iforest_anomaly_score
9,split_substation_based,train,isolation_forest,0.950,0.516082,2021,1380,641,0.652096,0.504002,0.647059,0.120125,0.202632,0.030435,iforest_anomaly_score


## 5. threshold / metrics / score / 모델 저장


In [8]:
threshold_rows = []
train_normal_scores = isolation_raw_score[train_normal_mask]
for quantile in THRESHOLD_QUANTILES:
    threshold_rows.append({
        'model_name': 'isolation_forest',
        'threshold_quantile': quantile,
        'threshold_value': threshold_lookup[('isolation_forest', quantile)],
        'train_normal_score_min': float(np.min(train_normal_scores)),
        'train_normal_score_median': float(np.median(train_normal_scores)),
        'train_normal_score_max': float(np.max(train_normal_scores)),
    })
thresholds_df = pd.DataFrame(threshold_rows)

model_metadata = {
    'model_version': MODEL_VERSION,
    'input_path': str(INPUT_PATH),
    'feature_count': len(selected_feature_columns),
    'selected_feature_columns': selected_feature_columns,
    'metadata_columns': metadata_columns,
    'training_filter': {
        'split_column': PRIMARY_SPLIT_COLUMN,
        'split_value': 'train',
        'label': 'normal',
    },
    'threshold_quantiles': THRESHOLD_QUANTILES,
    'default_threshold_quantile': DEFAULT_THRESHOLD_QUANTILE,
    'score_columns': {
        'isolation_forest': 'iforest_anomaly_score',
        'canonical_anomaly_score': 'anomaly_score',
    },
}

scores_df.to_csv(SCORES_PATH, index=False)
metrics_df.to_csv(METRICS_PATH, index=False)
thresholds_df.to_csv(THRESHOLDS_PATH, index=False)
threshold_sweep_metrics_df.to_csv(THRESHOLD_SWEEP_METRICS_PATH, index=False)

joblib.dump(scaler, SCALER_PATH)
joblib.dump(isolation_forest, IFOREST_MODEL_PATH)
MODEL_METADATA_PATH.write_text(json.dumps(model_metadata, ensure_ascii=False, indent=2), encoding='utf-8')

print('저장 완료 파일:')
print(SCORES_PATH)
print(METRICS_PATH)
print(THRESHOLDS_PATH)
print(THRESHOLD_SWEEP_METRICS_PATH)
print(SCALER_PATH)
print(IFOREST_MODEL_PATH)
print(MODEL_METADATA_PATH)

display(thresholds_df)


저장 완료 파일:
C:\Project3\HeatGrid_Agent\data\processed\ml_baseline\anomaly_baseline_scores.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_baseline\anomaly_baseline_metrics.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_baseline\anomaly_baseline_thresholds.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_baseline\anomaly_baseline_threshold_sweep_metrics.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_baseline\models\standard_scaler.joblib
C:\Project3\HeatGrid_Agent\data\processed\ml_baseline\models\isolation_forest.joblib
C:\Project3\HeatGrid_Agent\data\processed\ml_baseline\models\baseline_model_metadata.json


,model_name,threshold_quantile,threshold_value,train_normal_score_min,train_normal_score_median,train_normal_score_max
0,isolation_forest,0.950,0.516082,0.358841,0.423098,0.608249
1,isolation_forest,0.975,0.524479,0.358841,0.423098,0.608249
2,isolation_forest,0.990,0.535778,0.358841,0.423098,0.608249
